## 1. Môi trường và Cài đặt
Cài đặt COLMAP và clone repository 3D Gaussian Splatting để lấy script `convert.py`.

In [ ]:
!apt-get update && apt-get install -y colmap
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive

## 2. Cấu hình đường dẫn
Định nghĩa các đường dẫn tới dataset gốc và thư mục output.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

DATASET_ROOT = Path('/kaggle/input/viettel-ai-race-2026/dataset') # Thay đổi nếu cần
CONVERTED_ROOT = Path('/kaggle/working/converted_datasets')
REPO_DIR = Path('/kaggle/working/gaussian-splatting')

TRAIN_SPLITS = ['public_set'] # Thêm 'private_set' nếu có
TARGET_SCENES = None

# Cờ để quyết định có dọn dẹp các thư mục trung gian (input, distorted) để giảm dung lượng hay không
CLEANUP_INTERMEDIATE = True

## 3. Các hàm Tiền xử lý
Bao gồm hàm đọc camera, kiểm tra tính nhất quán, re-encode ảnh và undistort bằng COLMAP.

In [ ]:
import csv
import struct
from PIL import Image, ImageOps, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

CAMERA_MODELS = {
    0: ('SIMPLE_PINHOLE', 3), 1: ('PINHOLE', 4), 2: ('SIMPLE_RADIAL', 4),
    3: ('RADIAL', 5), 4: ('OPENCV', 8), 5: ('OPENCV_FISHEYE', 8),
    6: ('FULL_OPENCV', 12), 7: ('FOV', 5), 8: ('SIMPLE_RADIAL_FISHEYE', 4),
    9: ('RADIAL_FISHEYE', 5), 10: ('THIN_PRISM_FISHEYE', 12),
}

def read_camera_models(scene_path):
    cameras_bin = Path(scene_path) / 'sparse' / '0' / 'cameras.bin'
    models = []
    with cameras_bin.open('rb') as f:
        num = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num):
            camera_id, model_id, width, height = struct.unpack('<iiQQ', f.read(24))
            name, nparams = CAMERA_MODELS[model_id]
            params = struct.unpack('<' + 'd' * nparams, f.read(8 * nparams))
            models.append({'id': camera_id, 'model': name, 'width': width, 'height': height, 'params': params})
    return models

def prune_colmap_images_to_existing_files(scene_path):
    scene_path = Path(scene_path)
    images_bin = scene_path / 'sparse' / '0' / 'images.bin'
    images_dir = scene_path / 'images'
    existing = {p.name for p in images_dir.iterdir() if p.is_file()}
    kept, removed = [], []
    with images_bin.open('rb') as f:
        num_images = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num_images):
            fixed = f.read(64)
            name_bytes = bytearray()
            while True:
                ch = f.read(1)
                if ch == b'':
                    raise EOFError('Unexpected EOF while reading COLMAP image name')
                name_bytes += ch
                if ch == b'\x00':
                    break
            name = name_bytes[:-1].decode('utf-8')
            npoints_bytes = f.read(8)
            npoints = struct.unpack('<Q', npoints_bytes)[0]
            points_bytes = f.read(24 * npoints)
            record = fixed + bytes(name_bytes) + npoints_bytes + points_bytes
            if Path(name).name in existing:
                kept.append(record)
            else:
                removed.append(name)
    if removed:
        backup = images_bin.with_suffix('.bin.before_prune')
        if not backup.exists():
            shutil.copy2(images_bin, backup)
        with images_bin.open('wb') as f:
            f.write(struct.pack('<Q', len(kept)))
            for record in kept:
                f.write(record)
        print(f'Pruned images.bin: kept {len(kept)}/{len(kept) + len(removed)}, removed {len(removed)}')
    return len(kept), len(removed)

def find_challenge_scenes(root, splits):
    root = Path(root)
    scenes = []
    # Tìm kiếm tất cả test_poses.csv một cách linh hoạt dưới root
    for test_csv in sorted(root.rglob('**/test_poses.csv')):
        scene_dir = test_csv.parents[1]
        train_dir = scene_dir / 'train'
        
        # Xác định split của scene này
        split = scene_dir.parent.name
        if split not in splits:
            # Đề phòng trường hợp nested như phase1/public_set/HCM0204
            split = scene_dir.parent.parent.name
            
        if split in splits:
            if (train_dir / 'images').exists() and (train_dir / 'sparse' / '0' / 'cameras.bin').exists():
                scenes.append({
                    'split': split,
                    'scene_name': scene_dir.name,
                    'train_dir': train_dir,
                    'test_csv': test_csv,
                })
    return scenes

def read_colmap_image_names(scene_path):
    images_bin = Path(scene_path) / 'sparse' / '0' / 'images.bin'
    names = []
    with images_bin.open('rb') as f:
        num_images = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num_images):
            f.read(64)
            name_bytes = bytearray()
            while True:
                ch = f.read(1)
                if ch == b'':
                    raise EOFError('Unexpected EOF while reading COLMAP image name')
                if ch == b'\x00':
                    break
                name_bytes += ch
            npoints = struct.unpack('<Q', f.read(8))[0]
            f.seek(24 * npoints, 1)
            names.append(Path(name_bytes.decode('utf-8')).name)
    return names

def validate_scene_file_consistency(scene):
    train_dir = Path(scene['train_dir'])
    test_csv = Path(scene['test_csv'])
    test_images = test_csv.parent / 'images'
    train_files = {p.name for p in (train_dir / 'images').iterdir() if p.is_file()}
    test_files = {p.name for p in test_images.iterdir() if p.is_file()} if test_images.exists() else set()
    colmap_names = set(read_colmap_image_names(train_dir))
    missing_train = sorted(colmap_names - train_files)
    unregistered_train = sorted(train_files - colmap_names)
    with test_csv.open(newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    required_cols = {'image_name', 'qw', 'qx', 'qy', 'qz', 'tx', 'ty', 'tz', 'fx', 'fy', 'cx', 'cy', 'width', 'height'}
    missing_cols = required_cols - set(rows[0].keys() if rows else [])
    if missing_cols:
        raise RuntimeError(f"{scene['scene_name']}: test_poses.csv missing columns: {sorted(missing_cols)}")
    pose_names = {Path(row['image_name']).name for row in rows}
    missing_test = sorted(pose_names - test_files) if test_images.exists() else []
    extra_test = sorted(test_files - pose_names) if test_images.exists() else []
    test_files_label = len(test_files) if test_images.exists() else 'missing/optional'
    print(f"Check {scene['split']}/{scene['scene_name']}: train_files={len(train_files)}, colmap_records={len(colmap_names)}, test_files={test_files_label}, test_poses={len(rows)}")
    if missing_train:
        print(f"  WARN: COLMAP references {len(missing_train)} images not in train/images; prepare_scene_for_training() will prune them.")
    if unregistered_train:
        raise RuntimeError(f"{scene['scene_name']}: train/images has files not registered in COLMAP: {unregistered_train[:10]}")
    if missing_test:
        raise RuntimeError(f"{scene['scene_name']}: test_poses.csv references missing test images: {missing_test[:10]}")
    if extra_test:
        raise RuntimeError(f"{scene['scene_name']}: test/images has files not in test_poses.csv: {extra_test[:10]}")

def reencode_images_for_colmap(src_images, dst_input):
    dst_input.mkdir(parents=True, exist_ok=True)
    files = [p for p in Path(src_images).iterdir() if p.is_file()]
    print('Re-encoding images:', len(files))
    bad = []
    for idx, src in enumerate(files, 1):
        try:
            with Image.open(src) as im:
                im = ImageOps.exif_transpose(im).convert('RGB')
                im.save(dst_input / src.name, format='JPEG', quality=95, optimize=True)
        except Exception as exc:
            bad.append((src, exc))
        if idx % 50 == 0:
            print(f'  {idx}/{len(files)}')
    if bad:
        for src, exc in bad[:20]:
            print('Bad image:', src, repr(exc))
        raise RuntimeError(f'{len(bad)} images cannot be decoded')

def prepare_scene_for_training(scene):
    train_dir = Path(scene['train_dir'])
    model_names = {m['model'] for m in read_camera_models(train_dir)}
    if model_names <= {'PINHOLE', 'SIMPLE_PINHOLE'}:
        prune_colmap_images_to_existing_files(train_dir)
        return train_dir

    converted = CONVERTED_ROOT / scene['split'] / scene['scene_name'] / 'train'
    if (converted / 'images').exists() and (converted / 'sparse' / '0' / 'cameras.bin').exists():
        converted_models = {m['model'] for m in read_camera_models(converted)}
        if converted_models <= {'PINHOLE', 'SIMPLE_PINHOLE'}:
            print('Use converted:', converted)
            prune_colmap_images_to_existing_files(converted)
            return converted
        shutil.rmtree(converted)
    elif converted.exists():
        shutil.rmtree(converted)

    print('Convert scene:', scene['split'], scene['scene_name'])
    converted.mkdir(parents=True, exist_ok=True)
    reencode_images_for_colmap(train_dir / 'images', converted / 'input')
    shutil.copytree(train_dir / 'sparse', converted / 'distorted' / 'sparse', dirs_exist_ok=True)
    command = [
        'python', 'convert.py', '-s', str(converted), '--skip_matching'
    ]
    print('Running:', ' '.join(command))
    subprocess.check_call(command, cwd=REPO_DIR)
    prune_colmap_images_to_existing_files(converted)
    
    # Dọn dẹp thư mục trung gian để tiết kiệm không gian đĩa
    if CLEANUP_INTERMEDIATE:
        print(f"Cleaning up intermediate folders for {scene['scene_name']}...")
        input_dir = converted / 'input'
        distorted_dir = converted / 'distorted'
        if input_dir.exists():
            shutil.rmtree(input_dir)
        if distorted_dir.exists():
            shutil.rmtree(distorted_dir)
            
    return converted


## 4. Thực thi Tiền xử lý
Duyệt qua các scene và thực hiện convert.

In [ ]:
all_train_scenes = find_challenge_scenes(DATASET_ROOT, TRAIN_SPLITS)
if TARGET_SCENES is not None:
    all_train_scenes = [s for s in all_train_scenes if s['scene_name'] in TARGET_SCENES]
    found = {s['scene_name'] for s in all_train_scenes}
    missing = sorted(set(TARGET_SCENES) - found)
    if missing:
        raise RuntimeError(f'Missing target scenes: {missing}')
if not all_train_scenes:
    raise RuntimeError(f'No training scenes found for splits: {TRAIN_SPLITS}')

for scene in all_train_scenes:
    validate_scene_file_consistency(scene)

print('Train scenes to process:', [(s['split'], s['scene_name']) for s in all_train_scenes])

# Tiến hành preprocess
for scene in all_train_scenes:
    print(f"\n--- Processing {scene['split']} / {scene['scene_name']} ---")
    prepare_scene_for_training(scene)

print("\n✅ All scenes processed successfully.")

## 5. Đóng gói kết quả
Nén toàn bộ thư mục `converted_datasets` thành file zip để download hoặc upload thành một dataset Kaggle mới. Sau đó dọn dẹp các thư mục để giải phóng dung lượng cho thư mục /kaggle/working.

In [ ]:
import shutil
import os

# Nén dataset
zip_path = '/kaggle/working/converted_datasets.zip'
print(f"Zipping {CONVERTED_ROOT} to {zip_path}...")
shutil.make_archive('/kaggle/working/converted_datasets', 'zip', CONVERTED_ROOT)
print("✅ Zipping complete! Download /kaggle/working/converted_datasets.zip or upload it to Kaggle.")
